In [1]:
import os
import sqlite3
from sqlalchemy import create_engine, inspect, text
import pandas as pd

# Oracle connection
DB_HOST = os.getenv("DB_HOST", "localhost") 
DATABASE_URL = f"oracle+oracledb://hr:hr@{DB_HOST}:1521/?service_name=XEPDB1"

oracle_engine = create_engine(DATABASE_URL, pool_pre_ping=True)

# SQLite connection
sqlite_conn = sqlite3.connect('hr_database.db')

# Get all tables from Oracle HR schema
inspector = inspect(oracle_engine)
tables = inspector.get_table_names(schema='HR')

print(f"Found {len(tables)} tables: {tables}")

# Export each table
with oracle_engine.connect() as oracle_conn:
    for table in tables:
        print(f"Exporting {table}...")
        
        # Read from Oracle
        query = text(f'SELECT * FROM HR.{table}')
        df = pd.read_sql(query, oracle_conn)
        
        # Write to SQLite
        df.to_sql(table.lower(), sqlite_conn, if_exists='replace', index=False)
        
        print(f"  ✓ Exported {len(df)} rows")

sqlite_conn.close()
print("\n✓ Export complete! File: hr_database.db")

OperationalError: (oracledb.exceptions.OperationalError) DPY-6005: cannot connect to database (CONNECTION_ID=OvbNWu9LhDs0XYnvG6t/Eg==).
[WinError 10061] No connection could be made because the target machine actively refused it
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [23]:
import pandas as pd
import duckdb
from pathlib import Path

def export_database_to_excel(db_path: str, output_path: str = "database_export.xlsx"):
    """Export all tables from SQLite database to a single Excel file with multiple tabs"""
    
    conn = duckdb.connect()
    conn.execute(f"ATTACH '{db_path}' AS db (TYPE SQLITE)")
    
    # Get all tables
    tables = conn.execute("SHOW TABLES FROM db").fetchall()
    
    print(f"Found {len(tables)} tables. Exporting to Excel...")
    
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for (table_name,) in tables:
            print(f"  Exporting table: {table_name}")
            
            # Read entire table
            df = conn.execute(f"SELECT * FROM db.{table_name}").df()
            
            # Excel sheet names have max 31 characters
            sheet_name = table_name[:31]
            
            # Write to Excel tab
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    conn.close()
    print(f"✓ Export complete: {output_path}")
    print(f"  Total tables exported: {len(tables)}")


def export_database_to_csv(db_path: str, output_dir: str = "csv_export"):
    """Export all tables from SQLite database to separate CSV files"""
    
    # Create output directory
    Path(output_dir).mkdir(exist_ok=True)
    
    conn = duckdb.connect()
    conn.execute(f"ATTACH '{db_path}' AS db (TYPE SQLITE)")
    
    # Get all tables
    tables = conn.execute("SHOW TABLES FROM db").fetchall()
    
    print(f"Found {len(tables)} tables. Exporting to CSV...")
    
    for (table_name,) in tables:
        print(f"  Exporting table: {table_name}")
        
        # Read entire table
        df = conn.execute(f"SELECT * FROM db.{table_name}").df()
        
        # Save to CSV
        csv_path = Path(output_dir) / f"{table_name}.csv"
        df.to_csv(csv_path, index=False)
    
    conn.close()
    print(f"✓ Export complete: {output_dir}/")
    print(f"  Total files created: {len(tables)}")





# export_database_to_excel("./hr_database.db", "hr_export.xlsx")
export_database_to_csv("./hr_database.db", "hr_csv_export")

Found 7 tables. Exporting to CSV...
  Exporting table: countries
  Exporting table: departments
  Exporting table: employees
  Exporting table: job_history
  Exporting table: jobs
  Exporting table: locations
  Exporting table: regions
✓ Export complete: hr_csv_export/
  Total files created: 7


In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path="./hr_csv_export/countries.csv")

data = loader.load()

print(data)

[Document(metadata={'source': './hr_csv_export/countries.csv', 'row': 0}, page_content='country_id: AR\ncountry_name: Argentina\nregion_id: 2'), Document(metadata={'source': './hr_csv_export/countries.csv', 'row': 1}, page_content='country_id: AU\ncountry_name: Australia\nregion_id: 3'), Document(metadata={'source': './hr_csv_export/countries.csv', 'row': 2}, page_content='country_id: BE\ncountry_name: Belgium\nregion_id: 1'), Document(metadata={'source': './hr_csv_export/countries.csv', 'row': 3}, page_content='country_id: BR\ncountry_name: Brazil\nregion_id: 2'), Document(metadata={'source': './hr_csv_export/countries.csv', 'row': 4}, page_content='country_id: CA\ncountry_name: Canada\nregion_id: 2'), Document(metadata={'source': './hr_csv_export/countries.csv', 'row': 5}, page_content='country_id: CH\ncountry_name: Switzerland\nregion_id: 1'), Document(metadata={'source': './hr_csv_export/countries.csv', 'row': 6}, page_content='country_id: CN\ncountry_name: China\nregion_id: 3'), D

In [19]:
import duckdb

# Query in one line
result = duckdb.sql("SELECT * FROM sqlite_scan('hr_database.db', 'employees')").df()
print(result)

     employee_id first_name last_name     email  phone_number  hire_date  \
0            100     Steven      King     SKING  515.123.4567 2003-06-17   
1            101      Neena   Kochhar  NKOCHHAR  515.123.4568 2005-09-21   
2            102        Lex   De Haan   LDEHAAN  515.123.4569 2001-01-13   
3            103  Alexander    Hunold   AHUNOLD  590.423.4567 2006-01-03   
4            104      Bruce     Ernst    BERNST  590.423.4568 2007-05-21   
..           ...        ...       ...       ...           ...        ...   
102          202        Pat       Fay      PFAY  603.123.6666 2005-08-17   
103          203      Susan    Mavris   SMAVRIS  515.123.7777 2002-06-07   
104          204    Hermann      Baer     HBAER  515.123.8888 2002-06-07   
105          205    Shelley   Higgins  SHIGGINS  515.123.8080 2002-06-07   
106          206    William     Gietz    WGIETZ  515.123.8181 2002-06-07   

         job_id   salary  commission_pct  manager_id  department_id  
0       AD_PRES  

In [20]:
import duckdb

conn = duckdb.connect()
conn.execute("ATTACH 'hr_database.db' AS db (TYPE SQLITE)")

# Join employees and departments
result = conn.execute("""
    SELECT 
        e.first_name, 
        e.last_name, 
        e.salary,
        d.department_name
    FROM db.employees e
    JOIN db.departments d ON e.department_id = d.department_id
""")

print(result.df())

    first_name last_name   salary   department_name
0       Steven      King  24000.0         Executive
1        Neena   Kochhar  17000.0         Executive
2          Lex   De Haan  17000.0         Executive
3    Alexander    Hunold   9000.0                IT
4        Bruce     Ernst   6000.0                IT
..         ...       ...      ...               ...
101        Pat       Fay   6000.0         Marketing
102      Susan    Mavris   6500.0   Human Resources
103    Hermann      Baer  10000.0  Public Relations
104    Shelley   Higgins  12008.0        Accounting
105    William     Gietz   8300.0        Accounting

[106 rows x 4 columns]


In [ ]:
import duckdb

conn = duckdb.connect()
conn.execute("ATTACH 'hr_database.db' AS db (TYPE SQLITE)")


result = conn.execute("""
    SELECT 
        e.first_name, 
        e.last_name, 
        e.salary,
    FROM db.employees e
    INNER JOIN db.departments d ON e.department_id = d.department_id

""")

print(result.df())

    first_name last_name   salary
0       Steven      King  24000.0
1        Neena   Kochhar  17000.0
2          Lex   De Haan  17000.0
3    Alexander    Hunold   9000.0
4        Bruce     Ernst   6000.0
..         ...       ...      ...
101        Pat       Fay   6000.0
102      Susan    Mavris   6500.0
103    Hermann      Baer  10000.0
104    Shelley   Higgins  12008.0
105    William     Gietz   8300.0

[106 rows x 3 columns]


In [22]:
import duckdb

conn = duckdb.connect()
conn.execute("ATTACH 'hr_database.db' AS db (TYPE SQLITE)")

# Check what's in employees
print("EMPLOYEES:")
result = conn.execute("SELECT * FROM db.employees LIMIT 5").df()
print(result)

# Check what's in departments
print("\nDEPARTMENTS:")
result = conn.execute("SELECT * FROM db.departments LIMIT 5").df()
print(result)

# Check if department_id exists and has values
print("\nDEPARTMENT_IDs in employees:")
result = conn.execute("SELECT DISTINCT department_id FROM db.employees").df()
print(result)

print("\nDEPARTMENT_IDs in departments:")
result = conn.execute("SELECT DISTINCT department_id FROM db.departments").df()
print(result)

EMPLOYEES:
   employee_id first_name last_name     email  phone_number  hire_date  \
0          100     Steven      King     SKING  515.123.4567 2003-06-17   
1          101      Neena   Kochhar  NKOCHHAR  515.123.4568 2005-09-21   
2          102        Lex   De Haan   LDEHAAN  515.123.4569 2001-01-13   
3          103  Alexander    Hunold   AHUNOLD  590.423.4567 2006-01-03   
4          104      Bruce     Ernst    BERNST  590.423.4568 2007-05-21   

    job_id   salary  commission_pct  manager_id  department_id  
0  AD_PRES  24000.0             NaN         NaN           90.0  
1    AD_VP  17000.0             NaN       100.0           90.0  
2    AD_VP  17000.0             NaN       100.0           90.0  
3  IT_PROG   9000.0             NaN       102.0           60.0  
4  IT_PROG   6000.0             NaN       103.0           60.0  

DEPARTMENTS:
   department_id  department_name  manager_id  location_id
0             10   Administration       200.0         1700
1             20      

In [23]:
import duckdb

conn = duckdb.connect()
conn.execute("ATTACH 'hr_database.db' AS db (TYPE SQLITE)")

# Use LEFT JOIN to see all employees even without department match
result = conn.execute("""
    SELECT 
        e.first_name, 
        e.last_name, 
        e.salary,
        e.department_id as emp_dept_id,
        d.department_id as dept_dept_id,
        d.department_name
    FROM db.employees e
    LEFT JOIN db.departments d ON e.department_id = d.department_id
""").df()

print(result)

    first_name last_name   salary  emp_dept_id  dept_dept_id   department_name
0       Steven      King  24000.0         90.0            90         Executive
1        Neena   Kochhar  17000.0         90.0            90         Executive
2          Lex   De Haan  17000.0         90.0            90         Executive
3    Alexander    Hunold   9000.0         60.0            60                IT
4        Bruce     Ernst   6000.0         60.0            60                IT
..         ...       ...      ...          ...           ...               ...
102      Susan    Mavris   6500.0         40.0            40   Human Resources
103    Hermann      Baer  10000.0         70.0            70  Public Relations
104    Shelley   Higgins  12008.0        110.0           110        Accounting
105    William     Gietz   8300.0        110.0           110        Accounting
106  Kimberely     Grant   7000.0          NaN          <NA>              None

[107 rows x 6 columns]


In [1]:
from typing import List, Dict, Any
from decimal import Decimal
from sqlalchemy import text
from langchain_ollama import ChatOllama


def chat_once(prompt: str, client: ChatOllama) -> str:
    try:
        resp = client.invoke(prompt) 
        content = getattr(resp, "content", "")
        if isinstance(content, list):
            content = "".join(
                part.get("text", "") if isinstance(part, dict) else str(part)
                for part in content
            )
        return content or ""
    except Exception as e:
        print("Ollama call failed:", e)
        return ""

def translate_question(question: str, client: ChatOllama) -> str:
    """
    Use the LLM to translate any non-English question into clear English
    suitable for SQL querying. If the question is already English, keep it as-is.
    """

    prompt = f"""
You are a language detection and translation assistant.

Task:
- If the question is already in English, return it unchanged.
- If the question is NOT in English, translate it into clear, natural English.
- The final output MUST be a single English question suitable for SQL querying.
- Do NOT explain what you did.
- Do NOT add extra text.

Question:
{question}

English question:
"""

    english = chat_once(prompt, client).strip()
    if not english or len(english) < 5:
        print("LLM returned weak output, retrying...")
        english = chat_once(prompt, client).strip()

    print(f"Final English question: '{english}'")
    return english

def extract_oracle_schema(engine, schema="HR") -> str:
    query = f"""
    SELECT
        table_name,
        column_name,
        data_type
    FROM all_tab_columns
    WHERE owner = '{schema}'
    ORDER BY table_name, column_id
    """

    schema_dict = {}

    with engine.connect() as conn:
        rows = conn.exec_driver_sql(query).fetchall()

    for table, column, dtype in rows:
        schema_dict.setdefault(table, []).append(f"{column} {dtype}")

    schema_text = []
    for table, cols in schema_dict.items():
        schema_text.append(
            f"{table} ({', '.join(cols)})"
        )

    return "\n".join(schema_text)

def generate_sql(question: str, schema: str, client: ChatOllama) -> str:
    prompt = f"""
You are an expert Oracle SQL assistant for the HR database.

SCHEMA:
{schema}

IMPORTANT TABLES:
- EMPLOYEES (EMPLOYEE_ID, FIRST_NAME, LAST_NAME, SALARY, DEPARTMENT_ID, JOB_ID, HIRE_DATE)
- DEPARTMENTS (DEPARTMENT_ID, DEPARTMENT_NAME, LOCATION_ID)
- JOBS (JOB_ID, JOB_TITLE, MIN_SALARY, MAX_SALARY)
- LOCATIONS (LOCATION_ID, CITY, COUNTRY_ID)
- COUNTRIES (COUNTRY_ID, COUNTRY_NAME, REGION_ID)
- REGIONS (REGION_ID, REGION_NAME)

RULES:
- Use Oracle SQL syntax only.
- Use table aliases (employees e, departments d, jobs j).
- Always qualify columns with table aliases.
- Use FETCH FIRST N ROWS ONLY instead of LIMIT.
- For year extraction, use EXTRACT(YEAR FROM date_column).
- Use SYSDATE for current date.
- Return ONE valid Oracle SELECT query only.
- NO explanation, NO markdown.
- SELECT queries ONLY (NO INSERT, UPDATE, DELETE, DROP, CREATE, ALTER).
- Table and column names are UPPERCASE.
- DO NOT end the SQL statement with a semicolon (;).


Question:
{question}

SQL:
"""
    sql = chat_once(prompt, client).strip().strip("`")
    return sql

def is_safe_sql(sql: str) -> bool:
    """Block all DML/DDL - allow SELECT only for Oracle."""
    sql_upper = sql.strip().upper()
    
    dangerous = [
        'INSERT', 'UPDATE', 'DELETE', 'DROP', 'ALTER', 'CREATE', 
        'TRUNCATE', 'ATTACH', 'DETACH', 'REINDEX', 'ANALYZE',
        'PRAGMA', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT'
    ]
    
    for keyword in dangerous:
        if sql_upper.startswith(keyword):
            return False
    
    # Must start with SELECT
    return sql_upper.startswith('SELECT')

def ask_db(
    question: str,
    engine,                
    schema: str,
    client                 
) -> tuple[str, str, List[Dict[str, Any]]]:
    
    
    sql = ""
    rows_as_dict: List[Dict[str, Any]] = []
    message = "success"
    error_msg = None

    question = question.strip("“”\"").strip(".")

    for attempt in range(2):
        sql = generate_sql(question, schema, client).rstrip(";")
        print("Raw SQL from model:\n", sql)

        if not is_safe_sql(sql):
            message = "الاستعلام غير آمن ولا يمكن تنفيذه"
            return sql, message, []

        try:
            with engine.connect() as conn:
                result = conn.execute(text(sql))
                columns = result.keys()
                rows_as_dict = [
                    {col: float(val) if isinstance(val, Decimal) else val
                     for col, val in zip(columns, row)}
                    for row in result.fetchall()
                ]

            break  # success

        except Exception as e:
            error_msg = str(e)
            print("Execution failed:\n", error_msg)

            if attempt == 0:
                repair_prompt = f"""
You wrote this SQL:

{sql}

The Oracle database returned this error:
{error_msg}

Rewrite the query to fix the error.
Return ONLY valid Oracle SELECT SQL.
Do NOT use semicolons at the end.
"""
                sql = chat_once(repair_prompt, client).strip().rstrip(";")
            else:
                message = "حدث خطأ أثناء تنفيذ الاستعلام"
                return sql, message, []

    if not rows_as_dict:
        message = "لا توجد بيانات متاحة لهذا الطلب"
        return sql, message, []

    return sql, message, rows_as_dict


✓ Querying database: ./hr_database.db



CatalogException: Catalog Error: Table with name sqlite_master does not exist!
Did you mean "main.sqlite_master"?

LINE 1: SELECT name FROM db.sqlite_master WHERE type='table'
                         ^

In [29]:
import duckdb
import pandas as pd
from langchain_ollama import ChatOllama
from src.clients import build_client, build_translate_client


def chat_once(prompt: str, client: ChatOllama) -> str:
    try:
        resp = client.invoke(prompt) 
        content = getattr(resp, "content", "")
        if isinstance(content, list):
            content = "".join(
                part.get("text", "") if isinstance(part, dict) else str(part)
                for part in content
            )
        return content or ""
    except Exception as e:
        print("Ollama call failed:", e)
        return ""

def translate_question(question: str, client: ChatOllama) -> str:
    """
    Use the LLM to translate any non-English question into clear English
    suitable for SQL querying. If the question is already English, keep it as-is.
    """

    prompt = f"""
You are a language detection and translation assistant.

Task:
- If the question is already in English, return it unchanged.
- If the question is NOT in English, translate it into clear, natural English.
- The final output MUST be a single English question suitable for SQL querying.
- Do NOT explain what you did.
- Do NOT add extra text.

Question:
{question}

English question:
"""

    english = chat_once(prompt, client).strip()
    if not english or len(english) < 5:
        print("LLM returned weak output, retrying...")
        english = chat_once(prompt, client).strip()

    print(f"Final English question: '{english}'")
    return english


# def extract_db_schema(db_path: str) -> str:
#     """Extract schema from any SQLite database using DuckDB"""
#     conn = duckdb.connect()
#     conn.execute(f"ATTACH '{db_path}' AS db (TYPE SQLITE)")
    
#     # FIX: Use SHOW TABLES instead of querying sqlite_master
#     tables = conn.execute("SHOW TABLES FROM db").fetchall()
    
#     schema_dict = {}
    
#     for (table_name,) in tables:
#         # Get columns for each table
#         columns = conn.execute(f"PRAGMA db.table_info('{table_name}')").fetchall()
#         # col[1]=name, col[2]=type
#         schema_dict[table_name] = [f"{col[1]} {col[2]}" for col in columns]
    
#     # Format schema text
#     schema_text = []
#     for table, cols in schema_dict.items():
#         schema_text.append(f"{table} ({', '.join(cols)})")
    
#     conn.close()
#     return "\n".join(schema_text)


def extract_db_schema(db_path: str) -> str:
    """Extract schema from any SQLite database using DuckDB"""
    conn = duckdb.connect()
    conn.execute(f"ATTACH '{db_path}' AS db (TYPE SQLITE)")
    
    # Get all tables
    tables = conn.execute("SHOW TABLES FROM db").fetchall()
    
    schema_dict = {}
    
    for (table_name,) in tables:
        # Get a sample row to infer schema
        sample = conn.execute(f"SELECT * FROM db.{table_name} LIMIT 0").description
        schema_dict[table_name] = [f"{col[0]} {col[1]}" for col in sample]
    
    # Format schema text
    schema_text = []
    for table, cols in schema_dict.items():
        schema_text.append(f"{table} ({', '.join(cols)})")
    
    conn.close()
    return "\n".join(schema_text)

def generate_sql(question: str, schema: str, client: ChatOllama) -> str:
    """Generate SQL for any SQLite database using DuckDB syntax"""
    prompt = f"""
You are an expert SQL assistant using DuckDB syntax for SQLite databases.

SCHEMA:
{schema}

The database is attached as 'db'. Reference tables as: db.table_name

RULES:
- Use DuckDB/SQLite syntax
- Reference tables with 'db.' prefix: SELECT * FROM db.employees
- For joins: SELECT * FROM db.employees e JOIN db.departments d ON e.department_id = d.department_id
- Use LIMIT instead of FETCH FIRST for top N results
- Use strftime() for date operations
- Return ONE valid SELECT query only
- NO explanation, NO markdown, NO semicolons
- SELECT queries ONLY (NO INSERT, UPDATE, DELETE, DROP, CREATE, ALTER)
- Table and column names should match the schema (case-sensitive)

Question:
{question}

SQL:
"""
    sql = chat_once(prompt, client).strip().strip("`").strip(";")
    return sql


def query_any_db(db_path: str, question: str, translate_client: ChatOllama, sql_client: ChatOllama) -> pd.DataFrame:
    """Query any SQLite database"""
    
    print(f"✓ Querying database: {db_path}\n")
    
    # 1. Extract schema
    schema = extract_db_schema(db_path)
    print(f"Database Schema:\n{schema}\n")
    
    # 2. Translate question
    english_q = translate_question(question, translate_client)
    print(f"English question: {english_q}\n")
    
    # 3. Generate SQL
    sql = generate_sql(english_q, schema, sql_client)
    print(f"Generated SQL:\n{sql}\n")
    
    # 4. Execute
    try:
        conn = duckdb.connect()
        conn.execute(f"ATTACH '{db_path}' AS db (TYPE SQLITE)")
        result_df = conn.execute(sql).df()
        conn.close()
        
        print(f"Results ({len(result_df)} rows):")
        print(result_df)
        return result_df
        
    except Exception as e:
        print(f"Error executing SQL: {e}")
        print(f"Generated SQL was:\n{sql}")
        return pd.DataFrame()


# USAGE - works with ANY .db file
if __name__ == "__main__":
    client = build_client()
    translator_client = build_translate_client()
    
    # Works with any database!
    result = query_any_db(
        db_path="./hr_database.db",  
        question="#مين الموظفين اللي مرتباتهم أعلى من متوسط المرتبات في القسم بتاعهم؟",
        translate_client=translator_client,
        sql_client=client
    )

✓ Querying database: ./hr_database.db

Database Schema:
countries (country_id VARCHAR, country_name VARCHAR, region_id BIGINT)
departments (department_id BIGINT, department_name VARCHAR, manager_id DOUBLE, location_id BIGINT)
employees (employee_id BIGINT, first_name VARCHAR, last_name VARCHAR, email VARCHAR, phone_number VARCHAR, hire_date TIMESTAMP, job_id VARCHAR, salary DOUBLE, commission_pct DOUBLE, manager_id DOUBLE, department_id DOUBLE)
job_history (employee_id BIGINT, start_date TIMESTAMP, end_date TIMESTAMP, job_id VARCHAR, department_id BIGINT)
jobs (job_id VARCHAR, job_title VARCHAR, min_salary BIGINT, max_salary BIGINT)
locations (location_id BIGINT, street_address VARCHAR, postal_code VARCHAR, city VARCHAR, state_province VARCHAR, country_id VARCHAR)
regions (region_id BIGINT, region_name VARCHAR)

Final English question: 'What are the employees whose salaries are higher than the average salaries in their departments?'
English question: What are the employees whose salari

In [31]:
import requests

url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
response = requests.get(url)
with open("chinook.db", "wb") as f:
    f.write(response.content)
print("Downloaded chinook.db")

Downloaded chinook.db


In [ ]:
import duckdb
from src.clients import build_client,build_translate_client
import pandas as pd
from langchain_ollama import ChatOllama



def chat_once(prompt: str, client: ChatOllama) -> str:
    try:
        resp = client.invoke(prompt) 
        content = getattr(resp, "content", "")
        if isinstance(content, list):
            content = "".join(
                part.get("text", "") if isinstance(part, dict) else str(part)
                for part in content
            )
        return content or ""
    except Exception as e:
        print("Ollama call failed:", e)
        return ""

def translate_question(question: str, client: ChatOllama) -> str:
    """
    Use the LLM to translate any non-English question into clear English
    suitable for SQL querying. If the question is already English, keep it as-is.
    """

    prompt = f"""
You are a language detection and translation assistant.

Task:
- If the question is already in English, return it unchanged.
- If the question is NOT in English, translate it into clear, natural English.
- The final output MUST be a single English question suitable for SQL querying.
- Do NOT explain what you did.
- Do NOT add extra text.

Question:
{question}

English question:
"""

    english = chat_once(prompt, client).strip()
    if not english or len(english) < 5:
        print("LLM returned weak output, retrying...")
        english = chat_once(prompt, client).strip()

    print(f"Final English question: '{english}'")
    return english

def extract_db_schema(db_path: str) -> str:
    """Extract schema from any SQLite database using DuckDB"""
    conn = duckdb.connect()
    conn.execute(f"ATTACH '{db_path}' AS db (TYPE SQLITE)")
    
    # Get all tables
    tables = conn.execute("SELECT name FROM db.sqlite_master WHERE type='table'").fetchall()
    
    schema_dict = {}
    
    for (table_name,) in tables:
        # Get columns for each table
        columns = conn.execute(f"PRAGMA db.table_info('{table_name}')").fetchall()
        # col[1]=name, col[2]=type
        schema_dict[table_name] = [f"{col[1]} {col[2]}" for col in columns]
    
    # Format schema text
    schema_text = []
    for table, cols in schema_dict.items():
        schema_text.append(f"{table} ({', '.join(cols)})")
    
    conn.close()
    return "\n".join(schema_text)


def generate_sql(question: str, schema: str, client: ChatOllama) -> str:
    """Generate SQL for any SQLite database using DuckDB syntax"""
    prompt = f"""
You are an expert SQL assistant using DuckDB syntax for SQLite databases.

SCHEMA:
{schema}

The database is attached as 'db'. Reference tables as: db.table_name

RULES:
- Use DuckDB/SQLite syntax
- Reference tables with 'db.' prefix: SELECT * FROM db.employees
- For joins: SELECT * FROM db.employees e JOIN db.departments d ON e.department_id = d.department_id
- Use LIMIT instead of FETCH FIRST for top N results
- Use strftime() for date operations
- Return ONE valid SELECT query only
- NO explanation, NO markdown, NO semicolons
- SELECT queries ONLY (NO INSERT, UPDATE, DELETE, DROP, CREATE, ALTER)
- Table and column names should match the schema (case-sensitive)

Question:
{question}

SQL:
"""
    sql = chat_once(prompt, client).strip().strip("`").strip(";")
    return sql


def query_any_db(db_path: str, question: str, translate_client: ChatOllama, sql_client: ChatOllama) -> pd.DataFrame:
    """Query any SQLite database"""
    
    print(f"✓ Querying database: {db_path}\n")
    
    # 1. Extract schema
    schema = extract_db_schema(db_path)
    print(f"Database Schema:\n{schema}\n")
    
    # 2. Translate question
    english_q = translate_question(question, translate_client)
    print(f"English question: {english_q}\n")
    
    # 3. Generate SQL
    sql = generate_sql(english_q, schema, sql_client)
    print(f"Generated SQL:\n{sql}\n")
    
    # 4. Execute
    try:
        conn = duckdb.connect()
        conn.execute(f"ATTACH '{db_path}' AS db (TYPE SQLITE)")
        result_df = conn.execute(sql).df()
        conn.close()
        
        print(f"Results ({len(result_df)} rows):")
        print(result_df)
        return result_df
        
    except Exception as e:
        print(f"Error executing SQL: {e}")
        print(f"Generated SQL was:\n{sql}")
        return pd.DataFrame()



In [7]:
import os 
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

# Use absolute path to ensure we hit the correct file
db_path = os.path.abspath("./hr_database.db")

def extract_detailed_schema(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Database not found at: {path}")
    
    # We set sample_rows_in_table_info=3 so the LLM sees real data examples
    db_engine = SQLDatabase.from_uri(f"sqlite:///{path}", sample_rows_in_table_info=3)
    
    # This returns the CREATE TABLE statements + sample rows for ALL tables
    full_schema = db_engine.get_table_info()
    table_names = db_engine.get_usable_table_names()
    
    return full_schema, table_names

llm = build_client()
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

# 1. Get the detailed schema (with columns)
detailed_schema, actual_tables = extract_detailed_schema(db_path)

# 2. Update the prompt to be strictly column-aware
template = """You are a SQLite expert. You must generate a valid SQL query based on the schema provided below.

STRICT RULES:
1. ONLY use table names and column names found in the SCHEMA.
2. If the user asks for a filter (like "Sales"), use LOWER(column_name) = LOWER('value') to be safe.
3. Use the JOIN clauses if the information is spread across tables (e.g., employees and departments).
4. Return ONLY the raw SQL query.

SCHEMA:
{schema}

Top K: {top_k}
Table info: {table_info}
Question: {input}"""

prompt = PromptTemplate.from_template(template).partial(
    schema=detailed_schema,
    table_names=", ".join(actual_tables)
)

# 3. Build and Run Chain
chain = create_sql_query_chain(llm, db, prompt=prompt)

USER_QUESTION = "who has max salary?"

try:
    print(f" User Question: {USER_QUESTION}")
    raw_response = chain.invoke({"question": USER_QUESTION})
    
    # Clean SQL output
    sql_query = raw_response.replace("```sql", "").replace("```", "").strip()
    if "SELECT" in sql_query.upper():
        sql_query = sql_query[sql_query.upper().find("SELECT"):]
    
    print(f"SQL: {sql_query}")
    df = db.run(sql_query)
    print(f"Result: {db.run(sql_query)}")

except Exception as e:
    print(f" Error: {e}")

 User Question: who has max salary?
SQL: SELECT first_name, last_name FROM employees ORDER BY salary DESC LIMIT 1;
Result: [('Steven', 'King')]


In [8]:
print(df)

[('Steven', 'King')]


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama 
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat

# Initialize converter
converter = DocumentConverter(
    allowed_formats=[InputFormat.XLSX, InputFormat.CSV],
)

# Convert file
result = converter.convert("./hr_export.xlsx")

# Convert Docling output to LangChain documents
documents = []
docling_doc = result.document
markdown_content = docling_doc.export_to_markdown()

for table in docling_doc.tables:
    content = table.export_to_markdown(doc=docling_doc)
    documents.append(Document(
        page_content=content,
        metadata={"source": "hr_export.xlsx", "type": "table"}
    ))

if not documents:
    documents.append(Document(
        page_content=markdown_content,
        metadata={"source": "hr_export.xlsx"}
    ))

# Create embeddings (Keep using HuggingFace as it's efficient for local use)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vector store
vectorstore = FAISS.from_documents(documents, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# --- CHANGED: Set up ChatOllama ---
# Replace 'llama3' with your preferred local model (e.g., 'mistral', 'qwen2.5')
llm = ChatOllama(
    model="qwen2.5:3b-instruct", 
    temperature=0,
    # base_url="http://localhost:11434" # Default local Ollama URL
)

# Create prompt (Same as your original)
# Create a strict natural language prompt
prompt_template = """You are a helpful HR Assistant. 
Use the provided context (which contains table data) to answer the user's question in a clear, natural sentence.

STRICT RULES:
1. Do NOT provide SQL code, Python code, or any technical queries.
2. Answer only in plain text.
3. If the answer is not in the context, say "I don't have that information."
4. If there are multiple people with the same value, list them all.

Context: {context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)
# Create QA chain (Same logic)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

# Query
query = "who has the max salary?"
response = qa_chain.invoke({"query": query})
print(response['result'])

The employee with the maximum salary is identified by their job_id, which in this case is SA_MAN (Sales Manager). The corresponding max_salary for this job_id is $20080.

Here's a breakdown of the relevant information:

- Job ID: SA_MAN
- Job Title: Sales Manager
- Min Salary: $10000
- Max Salary: $20080

This can be seen in the "job_id" table, where there is only one row for SA_MAN with a max_salary of 20080.


In [34]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat



converter = DocumentConverter(
    allowed_formats=[InputFormat.XLSX,
                     InputFormat.CSV],
                     )

result = converter.convert("./hr_export.xlsx")


# Convert Docling output to LangChain documents
documents = []
docling_doc = result.document

# Export as markdown to get structured text
markdown_content = docling_doc.export_to_markdown()

# Split by tables or sections (Docling preserves structure)
# For now, create one document per page or table
# for table in docling_doc.tables:
#     content = table.export_to_markdown(doc=docling_doc)
#     print(content)
#     print("#" * 110)
#     documents.append(Document(
#         page_content=content,
#         metadata={"source": "hr_export.xlsx", "type": "table"}
#     ))

# Instead of raw table export, create "self-describing" rows
for table in docling_doc.tables:
    df = table.export_to_dataframe(doc=docling_doc)
    for _, row in df.iterrows():
        # Create a string like: "Employee ID: 101, Name: Steve King, Salary: 24000"
        row_str = ", ".join([f"{col}: {val}" for col, val in row.items()])
        documents.append(Document(
            page_content=row_str,
            metadata={"source": "hr_export.xlsx"}
        ))    

# If no tables, use the full markdown
if not documents:
    documents.append(Document(
        page_content=markdown_content,
        metadata={"source": "hr_export.xlsx"}
    ))

print(len(documents))
type(documents[0])



215


langchain_core.documents.base.Document

In [ ]:
# Create embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Create vector store (no chunking needed for CSV)
vectorstore = FAISS.from_documents(documents, embedding_model)
vectorstore.save_local("faiss_csv_index")

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Set up LLM
llm = ChatOllama(
    model="qwen2.5:3b-instruct", 
    temperature=0,
)
# Create prompt
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Answer based ONLY on the context below.
If not found, say "I don't have that information."

Context: {context}

Question: {question}

Answer:"""
)

# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)


query = "what is the manger_id of the marketing department?"
response = qa_chain.invoke({"query": query})
print(response['result'])

The manager_id of the Marketing department is 201.


In [35]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
import pandas as pd

converter = DocumentConverter(
    allowed_formats=[InputFormat.XLSX, InputFormat.CSV],
)

result = converter.convert("./hr_export.xlsx")
documents = []
docling_doc = result.document

# Export as markdown to get structured text (fallback)
markdown_content = docling_doc.export_to_markdown()

# **COLUMN-AWARE PROCESSING: Create hierarchical documents per table**
for table_ix, table in enumerate(docling_doc.tables):
    df = table.export_to_dataframe(doc=docling_doc)
    
    # 1. TABLE SCHEMA DOCUMENT (column names, types, sample)
    schema = f"""Table {table_ix}: {df.shape[0]} rows × {df.shape[1]} columns
Columns: {', '.join(df.columns.tolist())}
Sample: {dict(df.iloc[0].fillna('NULL')) if len(df)>0 else 'Empty'}"""
    documents.append(Document(
        page_content=schema,
        metadata={"source": "hr_export.xlsx", "type": "schema", "table_ix": table_ix}
    ))
    
    # 2. COLUMN AGGREGATIONS DOCUMENT (stats for SUM/AVG/COUNT)
    if len(df) > 0:
        agg_stats = df.describe(include='all').round(2).to_markdown(index=True)
        agg_doc = f"""Table {table_ix} Aggregations\n{agg_stats}"""
        documents.append(Document(
            page_content=agg_doc,
            metadata={"source": "hr_export.xlsx", "type": "aggregations", "table_ix": table_ix}
        ))
    
    # 3. FULL TABLE (small tables) OR SMART CHUNKS (large tables)
    if len(df) <= 100:
        # Full table markdown
        full_md = df.to_markdown(index=False)
        documents.append(Document(
            page_content=full_md,
            metadata={"source": "hr_export.xlsx", "type": "full_table", "table_ix": table_ix}
        ))
    else:
        # Chunked with column context preserved in EVERY chunk
        CHUNK_SIZE = 30
        for i in range(0, len(df), CHUNK_SIZE):
            chunk_df = df.iloc[i:i+CHUNK_SIZE].fillna('NULL')
            chunk_md = chunk_df.to_markdown(index=False)
            context = f"""Table {table_ix} rows {i+1}-{min(i+CHUNK_SIZE, len(df))} of {len(df)}
Columns: {', '.join(df.columns.tolist())}"""
            documents.append(Document(
                page_content=f"{context}\n{chunk_md}",
                metadata={
                    "source": "hr_export.xlsx",
                    "type": "table_chunk",
                    "table_ix": table_ix,
                    "row_start": i+1,
                    "row_end": min(i+CHUNK_SIZE, len(df)),
                    "total_rows": len(df)
                }
            ))

# Fallback if no tables found
if not documents:
    documents.append(Document(
        page_content=markdown_content,
        metadata={"source": "hr_export.xlsx", "type": "full_doc"}
    ))

print(f"Created {len(documents)} column-aware documents")
print(f"Document types: {set(doc.metadata.get('type') for doc in documents)}")
print("Sample:", type(documents[0]), documents[0].page_content[:200] + "...")


Created 24 column-aware documents
Document types: {'full_table', 'aggregations', 'table_chunk', 'schema'}
Sample: <class 'langchain_core.documents.base.Document'> Table 0: 25 rows × 3 columns
Columns: country_id, country_name, region_id
Sample: {'country_id': 'AR', 'country_name': 'Argentina', 'region_id': '2'}...


In [37]:
# Create embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Create vector store (no chunking needed for CSV)
vectorstore = FAISS.from_documents(documents, embedding_model)
vectorstore.save_local("faiss_csv_index")

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Set up LLM
llm = ChatOllama(
    model="qwen2.5:3b-instruct", 
    temperature=0,
)
# Create prompt
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Answer based ONLY on the context below.
If not found, say "I don't have that information."

Context: {context}

Question: {question}

Answer:"""
)

# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)


query = "when was steven king hired?"
response = qa_chain.invoke({"query": query})
print(response['result'])

Steven King, who is an employee in the HR schema and corresponds to the manager of the Finance department (Nancy Greenberg), was hired on August 17, 2002. This information can be found in the 'employees' table where Steven King's hire_date is set to '2002-08-17'.


In [12]:

from copyreg import pickle
import hashlib
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from transformers import AutoTokenizer







def rag_pipeline(file_path: str, question: str, llm_client):
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.do_table_structure = True
    
    converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.IMAGE,
            InputFormat.DOCX,
            InputFormat.HTML,
            InputFormat.PPTX,
            InputFormat.ASCIIDOC,
            InputFormat.CSV,
            InputFormat.MD,
            InputFormat.XLSX
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    result = converter.convert(file_path)
    doc = result.document
    # embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    # hf_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)

    embedding_model_name = "BAAI/bge-m3"
    hf_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
    

    chunker = HybridChunker(
        tokenizer=hf_tokenizer,
        # max_tokens=512, # type: ignore 
        # overlap=50   # type: ignore  
        max_tokens=1024,  # Sweet spot for BGE-M3 # type: ignore 
        overlap=128   # type: ignore 
)


    chunks = list(chunker.chunk(doc))
    # Convert to LangChain Documents
    documents = [
        Document(page_content=chunk.text, metadata=chunk.meta.export_json_dict())
        for chunk in chunks
    ]
    
    # Create embeddings and vector store
    # embedding_model = HuggingFaceEmbeddings(
    #     model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    # )
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={
            'device': 'cpu',  # or 'cuda' if GPU available
            'trust_remote_code': True
        },
        encode_kwargs={
            'normalize_embeddings': True,
            'batch_size': 32
        }
    )

    vectorstore = FAISS.from_documents(documents, embedding_model)
    
    # retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 5}
    )
    
    # QA prompt
    prompt = PromptTemplate.from_template("""Answer based ONLY on the context below.
If not found, say "I don't have that information."

Context: {context}

Question: {question}

Answer:""")
    
    # QA chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm_client,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True,
    )
    
    response = qa_chain.invoke({"query": question})
    return response['result']

       




                                                     

In [13]:
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
import os   

def build_llama_client():
    """Build Llama 3.1 8B client using Groq (RECOMMENDED)"""
    
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    
    if not GROQ_API_KEY:
        raise ValueError("Get API key from: https://console.groq.com/keys")
    
    return ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0,
        max_tokens=1024,
    )

llm_client = build_llama_client()

In [14]:
result = rag_pipeline('./noor_book.pdf', "What is the book about?", llm_client)

[INFO] 2026-02-15 08:42:01,124 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-15 08:42:01,127 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-15 08:42:01,154 [RapidOCR] download_file.py:60: File exists and is valid: D:\Injaz\text_sql_docker\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-15 08:42:01,155 [RapidOCR] main.py:50: Using D:\Injaz\text_sql_docker\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-15 08:42:01,479 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-15 08:42:01,479 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-15 08:42:01,479 [RapidOCR] download_file.py:60: File exists and is valid: D:\Injaz\text_sql_docker\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-02-15 08:42:01,479 [RapidOCR] main.py:50: Using D:\Injaz\text_sql_docker\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[I

In [7]:
result

"The context provided appears to be a collection of quotes and insights related to personal finance, wealth management, and career development. The content seems to revolve around financial advice, lessons learned from both successes and failures in business and investments, as well as reflections on education and the importance of having a growth mindset towards one's finances. It also touches upon the importance of taking calculated risks for personal growth and achieving financial success."

In [15]:
result

"I don't have that information."

In [ ]:
FROM python:3.11-slim

# Install system dependencies
RUN apt-get update && apt-get install -y \
    curl \
    libaio1t64 || apt-get install -y libaio1 \
    && rm -rf /var/lib/apt/lists/*

# Install uv using pip (avoids pulling the ghcr.io image)
RUN pip install uv

WORKDIR /app

# Enable bytecode compilation
ENV UV_COMPILE_BYTECODE=1

# Install dependencies
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-install-project

# Copy the rest of the code
COPY . .

# Final sync
RUN uv sync --frozen

EXPOSE 8000
EXPOSE 8501

In [ ]:
services:
  db:
    image: gvenzl/oracle-xe:latest
    environment:
      - ORACLE_PASSWORD=hr
      - APP_USER=hr
      - APP_USER_PASSWORD=hr
    ports:
      - "1521:1521"
    healthcheck:
      test: ["CMD", "healthcheck.sh"]
      interval: 10s
      timeout: 5s
      retries: 20

  api:
    build: .
    # This allows the container to talk to Ollama on your Windows host
    extra_hosts:
      - "host.docker.internal:host-gateway"
    environment:
      - DATABASE_URL=oracle+oracledb://hr:hr@db:1521/?service_name=XEPDB1
      # Point the API to the host machine for Ollama
      - OLLAMA_BASE_URL=http://host.docker.internal:11434
    command: uv run uvicorn main:app --host 0.0.0.0 --port 8000
    depends_on:
      db:
        condition: service_healthy
    ports:
      - "8000:8000"

In [ ]:
.venv
.git
__pycache__
*.pyc
.env

In [ ]:
# =============================================================
# Stages:
#   base        – system deps + uv + Python dependencies
#   backend     – FastAPI / uvicorn  (target: backend)
#   frontend    – Streamlit          (target: frontend)
# =============================================================


# ── Stage 1: base (shared) ────────────────────────────────
FROM python:3.11-slim AS base

RUN apt-get update && apt-get install -y \
    curl \
    && (apt-get install -y libaio1t64 || apt-get install -y libaio1) \
    && rm -rf /var/lib/apt/lists/*

RUN pip install uv

WORKDIR /app

ENV UV_COMPILE_BYTECODE=1

# Install deps first — cached as long as lockfile doesn't change
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-install-project --no-dev

# Copy source after deps are cached
COPY . .
RUN uv sync --frozen --no-dev


# ── Stage 2: backend ──────────────────────────────────────
FROM base AS backend

RUN mkdir -p data/files

EXPOSE 8000

CMD ["uv", "run", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]


# ── Stage 3: frontend ─────────────────────────────────────
FROM base AS frontend

EXPOSE 8501

CMD ["uv", "run", "streamlit", "run", "front.py", \
     "--server.address=0.0.0.0", \
     "--server.port=8501", \
     "--server.headless=true"]







     


In [ ]:
services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
      target: backend
    extra_hosts:
      - "host.docker.internal:host-gateway"
    environment:
      - OLLAMA_BASE_URL=http://host.docker.internal:11434
    ports:
      - "8000:8000"
    networks:
      - app-network

  frontend:
    build:
      context: .
      dockerfile: Dockerfile
      target: frontend
    ports:
      - "8501:8501"  # Streamlit default port
    environment:
      - API_URL=http://api:8000  # Frontend can call API by service name
    depends_on:
      - api
    networks:
      - app-network

networks:
  app-network:
    driver: bridge